# RQ1 Evaluation Suite: Document Retrieval Performance

This notebook implements the complete, methodologically rigorous experimental pipeline for evaluating **Research Question 1 (RQ1)**. 

### Evaluator-Approved Safeguards:
1. **Corpus Preflight Assertions:** Verifies that all 40 PDFs, extracted JSON files, generated notes, and gold labels map to the exact same document IDs.
2. **Strict Graph Leakage Prevention:** Uses a title map to detect paper-to-paper links and strips them completely, while converting general concept/topic links to plain text.
3. **Automatic 19-Minute Cache Bypass:** Bypasses baseline embedding generation if the file already exists on disk.
4. **Five-Fold Cross-Validated Alpha Selection:** Selects the graph-boosting coefficient $\alpha$ using deterministic score-transition candidates and 5-fold held-out evaluation.

In [ ]:
import json
import re
import math
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading SentenceTransformer model ('nomic-ai/nomic-embed-text-v1.5')...")
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
print(f"Model Max Sequence Length: {model.max_seq_length} tokens")

Loading SentenceTransformer model ('nomic-ai/nomic-embed-text-v1.5')...


<All keys matched successfully>


Model Max Sequence Length: 8192 tokens


In [ ]:
def build_title_map(notes_dir: Path) -> dict[str, str]:
    """
    Builds a lowercase Title/Alias/ArXivID -> arXiv ID mapping.
    This is used to detect paper-to-paper references.
    """
    title_map = {}
    for note_path in notes_dir.glob("*.md"):
        doc_id = note_path.stem
        content = note_path.read_text(encoding="utf-8", errors="ignore")
        
        # Lowercase arXiv ID mapping
        title_map[doc_id.lower()] = doc_id
        
        # Extract title from frontmatter
        title_match = re.search(r'^title:\s*"(.*?)"', content, re.MULTILINE)
        if not title_match:
            title_match = re.search(r'^title:\s*(.*?)$', content, re.MULTILINE)
        if title_match:
            title_map[title_match.group(1).strip().strip('"').strip("'").lower()] = doc_id
            
        # Extract aliases from frontmatter
        aliases_match = re.search(r'^aliases:\s*\[(.*?)\]', content, re.MULTILINE)
        if aliases_match:
            aliases = [a.strip().strip('"').strip("'") for a in aliases_match.group(1).split(",") if a.strip()]
            for alias in aliases:
                title_map[alias.lower()] = doc_id
    return title_map

def clean_markdown_links(text: str, title_map: dict[str, str]) -> str:
    """
    Strips Obsidian WikiLinks from Markdown notes to prevent relation leakages:
    - Paper-to-paper links (targets matching our 40 papers) are stripped completely.
    - Concept/topic links (targets not in our paper corpus) are converted to plain text.
    """
    def replacer(match):
        raw_target = match.group(1).split('|')[0].strip()
        display_text = match.group(1).split('|')[-1].strip()
        
        # If the target refers to another paper in our corpus, strip it entirely
        if raw_target.lower() in title_map:
            return ""
        # Otherwise, keep the display text as plain text
        return display_text

    return re.sub(r"\[\[([^\]]+)\]\]", replacer, text)

def split_text_by_tokens(text: str, tokenizer, chunk_size: int = 512, overlap: int = 50) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(len(tokens), start + chunk_size)
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
        if end == len(tokens):
            break
        start = max(0, end - overlap)
    return chunks

In [ ]:
def run_preflight_check(extracted_dir: Path, notes_dir: Path, queries_path: Path, gold_path: Path):
    """
    Verifies that all 40 PDFs, extracted JSON files, generated notes, and 
    gold labels map to the exact same document IDs.
    """
    extracted_ids = {f.stem for f in extracted_dir.glob("*.json") if not f.name.startswith("_")}
    notes_ids = {f.stem for f in notes_dir.glob("*.md")}
    
    with open(queries_path, encoding="utf-8") as f:
        queries = json.load(f)
    
    with open(gold_path, encoding="utf-8") as f:
        gold = json.load(f)
        
    query_ids = [q["query_id"] for q in queries]
    unique_query_ids = set(query_ids)
    
    gold_ids = set()
    for doc_list in gold.values():
        gold_ids.update(doc_list)
        
    print(f"Preflight Check:")
    print(f"  Extracted JSON papers count: {len(extracted_ids)}")
    print(f"  Generated notes count:       {len(notes_ids)}")
    print(f"  Total queries count:         {len(queries)}")
    print(f"  Unique query IDs count:      {len(unique_query_ids)}")
    print(f"  Gold relevance papers count:  {len(gold_ids)}")
    
    assert len(extracted_ids) == 40, f"Error: Expected 40 extracted texts, found {len(extracted_ids)}"
    assert extracted_ids == notes_ids, "Error: Extracted JSON IDs and Generated Note IDs mismatch!"
    assert len(queries) == 40, f"Error: Expected exactly 40 queries, found {len(queries)}"
    assert len(unique_query_ids) == len(queries), "Error: Query IDs are not unique!"
    assert set(gold.keys()) == unique_query_ids, "Error: Mismatch between query IDs and gold label keys!"
    assert gold_ids.issubset(extracted_ids), "Error: Gold labels reference paper IDs not present in corpus!"
    print("PASS: Preflight corpus check completed successfully. All 40 papers and queries are aligned.")

In [ ]:
def build_baseline_index(extracted_dir: Path, output_path: Path):
    json_files = sorted(extracted_dir.glob("*.json"))
    docs = []
    
    for f in json_files:
        if f.name.startswith("_"):
            continue
        data = json.loads(f.read_text(encoding="utf-8"))
        text = data["text"]
        
        chunks = split_text_by_tokens(text, model.tokenizer, chunk_size=512, overlap=50)
        for i, chunk in enumerate(chunks):
            docs.append({
                "doc_id": data["doc_id"],
                "source": data["source_pdf"],
                "chunk_id": i,
                "text": chunk
            })
            
    prefixed_texts = ["search_document: " + d["text"] for d in docs]
    print(f"Embedding {len(docs)} chunks for Pipeline A...")
    embeddings = model.encode(prefixed_texts, convert_to_numpy=True, normalize_embeddings=True)
    
    index_data = {
        "docs": docs,
        "embeddings": embeddings.tolist(),
        "model_name": "nomic-ai/nomic-embed-text-v1.5"
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f_out:
        json.dump(index_data, f_out)
    print("Pipeline A baseline index built successfully.")

def build_structured_index(notes_dir: Path, output_path: Path):
    note_files = sorted(notes_dir.glob("*.md"))
    doc_ids = []
    texts = []
    max_limit = model.max_seq_length
    
    # Pre-build title map to detect paper links
    title_map = build_title_map(notes_dir)
    
    for md_path in note_files:
        content = md_path.read_text(encoding="utf-8")
        cleaned_text = clean_markdown_links(content, title_map)
        
        # Length check
        tokens = model.tokenizer.encode(cleaned_text, add_special_tokens=False)
        token_len = len(tokens)
        if token_len > max_limit:
            raise ValueError(f"Error: Note {md_path.name} has {token_len} tokens, exceeding {max_limit}!")
            
        doc_ids.append(md_path.stem)
        texts.append(cleaned_text)
        
    prefixed_texts = ["search_document: " + t for t in texts]
    print(f"Embedding {len(doc_ids)} structured notes for Pipeline B...")
    embeddings = model.encode(prefixed_texts, convert_to_numpy=True, normalize_embeddings=True)
    
    index_data = {
        "doc_ids": doc_ids,
        "embeddings": embeddings.tolist(),
        "model": "nomic-ai/nomic-embed-text-v1.5"
    }
    with open(output_path, "w", encoding="utf-8") as f_out:
        json.dump(index_data, f_out)
    print("Pipeline B structured index built successfully.")

In [ ]:
def retrieve_baseline_maxp(query: str, docs, embeddings, k: int = 5) -> list[str]:
    query_text = "search_query: " + query
    query_emb = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
    similarities = cosine_similarity(query_emb, embeddings)[0]
    
    doc_max_scores = {}
    for idx, score in enumerate(similarities):
        doc_id = docs[idx]["doc_id"]
        if doc_id not in doc_max_scores or score > doc_max_scores[doc_id]:
            doc_max_scores[doc_id] = float(score)
            
    sorted_docs = sorted(doc_max_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_docs[:k]]

def retrieve_structured(query: str, index_data, k: int = 5) -> list[str]:
    query_text = "search_query: " + query
    query_emb = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
    doc_embs = np.array(index_data["embeddings"])
    similarities = cosine_similarity(query_emb, doc_embs)[0]
    top_indices = np.argsort(similarities)[::-1][:k]
    return [index_data["doc_ids"][i] for i in top_indices]

def retrieve_link_expanded(query: str, index_data, notes_dir: Path, k: int = 5, alpha: float = 0.05) -> list[str]:
    query_text = "search_query: " + query
    query_emb = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
    doc_ids = index_data["doc_ids"]
    doc_embs = np.array(index_data["embeddings"])
    
    similarities = cosine_similarity(query_emb, doc_embs)[0]
    doc_scores = {doc_ids[i]: float(similarities[i]) for i in range(len(doc_ids))}
    
    initial_seeds = sorted(doc_ids, key=lambda x: doc_scores[x], reverse=True)[:k]
    title_map = build_title_map(notes_dir)
    linked_docs = set()
    
    for doc_id in initial_seeds:
        note_path = notes_dir / f"{doc_id}.md"
        if note_path.exists():
            text = note_path.read_text(encoding="utf-8")
            raw_links = re.findall(r"\[\[([^\]]+)\]\]", text)
            targets = [r.split("|")[0].strip() for r in raw_links]
            for target in targets:
                matched_id = title_map.get(target.lower())
                if matched_id and matched_id in doc_scores and matched_id not in initial_seeds:
                    linked_docs.add(matched_id)
                    
    boosted_scores = dict(doc_scores)
    for l_doc in linked_docs:
        boosted_scores[l_doc] += alpha
        
    re_ranked = sorted(doc_ids, key=lambda x: boosted_scores[x], reverse=True)[:k]
    return re_ranked

In [ ]:
def recall_at_5(retrieved, relevant) -> float:
    if not relevant:
        return 0.0
    hits = sum(1 for d in retrieved[:5] if d in relevant)
    return hits / len(relevant)

def precision_at_5(retrieved, relevant) -> float:
    hits = sum(1 for d in retrieved[:5] if d in relevant)
    return hits / 5.0

def mrr_at_5(retrieved, relevant) -> float:
    for rank, doc_id in enumerate(retrieved[:5], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def run_eval_suite(queries, gold, baseline_index_path, structured_index_path, notes_dir, output_dir):
    with open(baseline_index_path, encoding="utf-8") as f:
        baseline_data = json.load(f)
    baseline_docs = baseline_data["docs"]
    baseline_embs = np.array(baseline_data["embeddings"])
    
    with open(structured_index_path, encoding="utf-8") as f:
        structured_data = json.load(f)
        
    runs = {"baseline": {}, "structured": {}}
    per_query_metrics = {}
    
    for q in queries:
        qid = q["query_id"]
        text = q["text"]
        rel = set(gold[qid])
        
        # Search Pipelines A & B
        runs["baseline"][qid] = retrieve_baseline_maxp(text, baseline_docs, baseline_embs, k=5)
        runs["structured"][qid] = retrieve_structured(text, structured_data, k=5)
        
        assert len(runs["baseline"][qid]) == 5
        assert len(runs["structured"][qid]) == 5
        
        # Compute metrics per query
        per_query_metrics[qid] = {}
        for name in runs.keys():
            retrieved = runs[name][qid]
            per_query_metrics[qid][name] = {
                "retrieved": retrieved,
                "gold": list(rel),
                "Recall@5": recall_at_5(retrieved, rel),
                "Precision@5": precision_at_5(retrieved, rel),
                "MRR@5": mrr_at_5(retrieved, rel)
            }
            
    # Compute macro averages
    macro_results = {}
    for run_name in runs.keys():
        recalls = [per_query_metrics[qid][run_name]["Recall@5"] for qid in per_query_metrics.keys()]
        precisions = [per_query_metrics[qid][run_name]["Precision@5"] for qid in per_query_metrics.keys()]
        mrrs = [per_query_metrics[qid][run_name]["MRR@5"] for qid in per_query_metrics.keys()]
        
        macro_results[run_name] = {
            "Recall@5": float(np.mean(recalls)),
            "Precision@5": float(np.mean(precisions)),
            "MRR@5": float(np.mean(mrrs))
        }
        
    # Save reports to disk
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_dir / "baseline_run.json", "w", encoding="utf-8") as f:
        json.dump(runs["baseline"], f, indent=2)
    with open(output_dir / "structured_run.json", "w", encoding="utf-8") as f:
        json.dump(runs["structured"], f, indent=2)
    with open(output_dir / "per_query_report.json", "w", encoding="utf-8") as f:
        json.dump(per_query_metrics, f, indent=2)
    with open(output_dir / "retrieval_metrics.json", "w", encoding="utf-8") as f:
        json.dump(macro_results, f, indent=2)
        
    return macro_results

In [ ]:
extracted_dir = Path("data/extracted_text")
notes_dir = Path("data/generated_notes")
baseline_idx = Path("data/results/baseline_index.json")
structured_idx = Path("data/results/structured_index.json")
queries_path = Path("data/queries/queries.json")
gold_path = Path("data/gold_labels/gold_labels.json")
output_results_dir = Path("data/results")

# 1. Create results directory before building
output_results_dir.mkdir(parents=True, exist_ok=True)

# 2. Run Preflight Corpus Validation Check
run_preflight_check(extracted_dir, notes_dir, queries_path, gold_path)

# 3. Build Indices (Bypass baseline embedding calculation if already exists)
if not baseline_idx.exists():
    build_baseline_index(extracted_dir, baseline_idx)
else:
    print("Baseline index already exists on disk. Skipping baseline embedding generation (saves 19 minutes).")
    
# Always rebuild structured index since notes were recently cleaned on disk
build_structured_index(notes_dir, structured_idx)

# 4. Load queries and gold labels
with open(queries_path, encoding="utf-8") as f:
    queries = json.load(f)
with open(gold_path, encoding="utf-8") as f:
    gold = json.load(f)
    
# 5. Run Metrics Suite (writes Pipeline A & B metrics to data/results/)
eval_scores = run_eval_suite(queries, gold, baseline_idx, structured_idx, notes_dir, output_results_dir)

# 6. Print Pipelines A & B comparison table
print("\n" + "=" * 45)
print(f"{'Metric':<18} | {'Baseline RAG':<12} | {'Structured RAG':<12}")
print("=" * 45)
for metric in ["Recall@5", "Precision@5", "MRR@5"]:
    b = eval_scores["baseline"][metric]
    s = eval_scores["structured"][metric]
    print(f"{metric:<18} | {b:<12.4f} | {s:<12.4f}")
print("=" * 45)

Preflight Check:
  Extracted JSON papers count: 40
  Generated notes count:       40
  Total queries count:         40
  Unique query IDs count:      40
  Gold relevance papers count:  25
PASS: Preflight corpus check completed successfully. All 40 papers and queries are aligned.
Baseline index already exists on disk. Skipping baseline embedding generation (saves 19 minutes).
Embedding 40 structured notes for Pipeline B...
Pipeline B structured index built successfully.

Metric             | Baseline RAG | Structured RAG
Recall@5           | 0.6958       | 0.6958      
Precision@5        | 0.2350       | 0.2250      
MRR@5              | 0.7292       | 0.7438      


## 6. Five-Fold Cross-Validated Alpha Selection (Pipeline C)

We optimize the link-boosting parameter $\alpha$ using a **five-fold held-out evaluation with fold-specific alpha selection**. We use `StratifiedKFold` to build representative, category-balanced train and held-out test splits, preventing domain concentration biases.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from collections import Counter

# Load structured index
with open(structured_idx, encoding="utf-8") as f:
    structured_data = json.load(f)
title_map = build_title_map(notes_dir)

# 1. Pre-calculate similarity vectors and cache link indicators for all 40 queries
query_embs = {}
cached_scores = {}
cached_links = {}

doc_ids = structured_data["doc_ids"]
doc_embs = np.array(structured_data["embeddings"])

print("Caching query embeddings, cosine similarities, and seed link sets...")
for q in queries:
    qid = q["query_id"]
    prefixed = "search_query: " + q["text"]
    q_emb = model.encode([prefixed], convert_to_numpy=True, normalize_embeddings=True)
    query_embs[qid] = q_emb
    
    # Cosine similarities
    similarities = cosine_similarity(q_emb, doc_embs)[0]
    doc_scores = {doc_ids[i]: float(similarities[i]) for i in range(len(doc_ids))}
    cached_scores[qid] = doc_scores
    
    # Initial top-5 seeds
    seeds = sorted(doc_ids, key=lambda x: doc_scores[x], reverse=True)[:5]
    
    # Determine linked docs outside seeds
    linked = set()
    for doc_id in seeds:
        note_path = notes_dir / f"{doc_id}.md"
        if note_path.exists():
            text = note_path.read_text(encoding="utf-8")
            raw_links = re.findall(r"\[\[([^\]]+)\]\]", text)
            targets = [r.split("|")[0].strip() for r in raw_links]
            for target in targets:
                matched_id = title_map.get(target.lower())
                if matched_id and matched_id in doc_scores and matched_id not in seeds:
                    linked.add(matched_id)
    cached_links[qid] = linked

# Helper: Candidate generator for score transitions (train queries only)
def get_alpha_candidates(train_queries_subset):
    candidates = {0.0}
    for q in train_queries_subset:
        qid = q["query_id"]
        doc_scores = cached_scores[qid]
        seeds = sorted(doc_ids, key=lambda x: doc_scores[x], reverse=True)[:5]
        linked = cached_links[qid]
        
        for u in seeds:
            s_u = doc_scores[u]
            for l in linked:
                s_l = doc_scores[l]
                if s_u > s_l:
                    diff = (s_u - s_l) + 1e-8
                    candidates.add(round(diff, 8))
    return sorted(list(candidates))

# Helper: Evaluate a candidate alpha on a queries subset
def evaluate_alpha(queries_subset, alpha):
    recalls = []
    precisions = []
    mrrs = []
    
    for q in queries_subset:
        qid = q["query_id"]
        rel = set(gold[qid])
        doc_scores = cached_scores[qid]
        linked = cached_links[qid]
        
        boosted_scores = dict(doc_scores)
        for l in linked:
            boosted_scores[l] += alpha
            
        retrieved = sorted(doc_ids, key=lambda x: boosted_scores[x], reverse=True)[:5]
        
        recalls.append(recall_at_5(retrieved, rel))
        precisions.append(precision_at_5(retrieved, rel))
        mrrs.append(mrr_at_5(retrieved, rel))
        
    return np.mean(recalls), np.mean(precisions), np.mean(mrrs)

# Helper: Optimal alpha selection prioritized: MRR -> Recall -> Precision -> min alpha
def select_optimal_alpha(queries_subset, candidates):
    best_score = (-1.0, -1.0, -1.0, 999.0)
    best_alpha = 0.0
    best_metrics = (0.0, 0.0, 0.0)
    
    for alpha in candidates:
        rec, prec, mrr = evaluate_alpha(queries_subset, alpha)
        # Priority score sorting: MRR -> Recall -> Precision -> smallest alpha
        score_tuple = (mrr, rec, prec, -alpha)
        if score_tuple > best_score:
            best_score = score_tuple
            best_alpha = alpha
            best_metrics = (rec, prec, mrr)
            
    return best_alpha, best_metrics

Caching query embeddings, cosine similarities, and seed link sets...


In [ ]:
# 2. Stratified 5-Fold Splitting
categories = [q["category"] for q in queries]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_report = {}
out_of_fold_predictions = {}
all_held_out_query_ids = []

# Assertions arrays to verify properties
fold_held_out_sets = []
simple_counts = []
comp_counts = []
nesy_counts = []

print("\n--- Running Five-Fold Cross-Validated Alpha Selection (Stratified) ---")
for fold, (train_indices, test_indices) in enumerate(skf.split(np.zeros(len(queries)), categories)):
    train_queries = [queries[i] for i in train_indices]
    test_queries = [queries[i] for i in test_indices]
    
    train_ids = [q["query_id"] for q in train_queries]
    test_ids = [q["query_id"] for q in test_queries]
    all_held_out_query_ids.extend(test_ids)
    
    # Category counts validation
    train_cats = Counter([q["category"] for q in train_queries])
    test_cats = Counter([q["category"] for q in test_queries])
    
    simple_counts.append(test_cats.get("simple_factual", 0))
    comp_counts.append(test_cats.get("comparison", 0))
    nesy_counts.append(test_cats.get("neurosymbolic", 0))
    
    # Assertions 1 & 2: overlaps and boundaries
    assert not set(train_ids).intersection(set(test_ids)), f"Overlap in Fold {fold}!"
    fold_held_out_sets.append(set(test_ids))
    
    # Generate candidates from training queries only
    candidates = get_alpha_candidates(train_queries)
    
    # Select optimal alpha on training queries
    selected_alpha, train_metrics = select_optimal_alpha(train_queries, candidates)
    
    # Evaluate on held-out test queries
    fold_recalls = []
    fold_precisions = []
    fold_mrrs = []
    
    for q in test_queries:
        qid = q["query_id"]
        rel = set(gold[qid])
        doc_scores = cached_scores[qid]
        linked = cached_links[qid]
        
        boosted_scores = dict(doc_scores)
        for l in linked:
            boosted_scores[l] += selected_alpha
            
        retrieved = sorted(doc_ids, key=lambda x: boosted_scores[x], reverse=True)[:5]
        out_of_fold_predictions[qid] = retrieved
        
        fold_recalls.append(recall_at_5(retrieved, rel))
        fold_precisions.append(precision_at_5(retrieved, rel))
        fold_mrrs.append(mrr_at_5(retrieved, rel))
        
    cv_report[f"fold_{fold}"] = {
        "train_queries": train_ids,
        "held_out_queries": test_ids,
        "train_category_counts": dict(train_cats),
        "held_out_category_counts": dict(test_cats),
        "candidate_alphas_count": len(candidates),
        "selected_alpha": selected_alpha,
        "train_metrics": {
            "Recall@5": train_metrics[0],
            "Precision@5": train_metrics[1],
            "MRR@5": train_metrics[2]
        },
        "held_out_metrics": {
            "Recall@5": float(np.mean(fold_recalls)),
            "Precision@5": float(np.mean(fold_precisions)),
            "MRR@5": float(np.mean(fold_mrrs))
        }
    }
    print(f"Fold {fold}: Alpha = {selected_alpha:.6f} | Cand = {len(candidates)} | Held-out Recall = {np.mean(fold_recalls):.4f} | MRR = {np.mean(fold_mrrs):.4f}")

# Assertions 1 & 4 check: query splits uniqueness and category balance
unique_held_outs = set().union(*fold_held_out_sets)
assert len(unique_held_outs) == 40, "Queries count mismatch in splits!"
for i in range(5):
    for j in range(i + 1, 5):
        assert not fold_held_out_sets[i].intersection(fold_held_out_sets[j]), "Queries duplicate across folds!"

assert max(simple_counts) - min(simple_counts) <= 1, "Simple factual counts unbalanced!"
assert max(comp_counts) - min(comp_counts) <= 1, "Comparison counts unbalanced!"
assert max(nesy_counts) - min(nesy_counts) <= 1, "Neurosymbolic counts unbalanced!"
print("PASS: Folds construct assertions verified. Query categories are balanced and split deterministically.")


--- Running Five-Fold Cross-Validated Alpha Selection (Stratified) ---
Fold 0: Alpha = 0.037861 | Cand = 1021 | Held-out Recall = 0.9583 | MRR = 1.0000
Fold 1: Alpha = 0.037861 | Cand = 1006 | Held-out Recall = 0.6458 | MRR = 0.6875
Fold 2: Alpha = 0.008587 | Cand = 936 | Held-out Recall = 0.5625 | MRR = 0.6250
Fold 3: Alpha = 0.037861 | Cand = 1031 | Held-out Recall = 0.7500 | MRR = 0.7500
Fold 4: Alpha = 0.037861 | Cand = 1011 | Held-out Recall = 0.6771 | MRR = 0.6500
PASS: Folds construct assertions verified. Query categories are balanced and split deterministically.


In [ ]:
# 3. Aggregate out-of-fold predictions and calculate final scores
aggregated_recalls = []
aggregated_precisions = []
aggregated_mrrs = []
link_run_results = {}

for q in queries:
    qid = q["query_id"]
    rel = set(gold[qid])
    retrieved = out_of_fold_predictions[qid]
    link_run_results[qid] = retrieved
    
    aggregated_recalls.append(recall_at_5(retrieved, rel))
    aggregated_precisions.append(precision_at_5(retrieved, rel))
    aggregated_mrrs.append(mrr_at_5(retrieved, rel))
    
# Assertion 3: verify aggregated size
assert len(link_run_results) == 40, "Aggregated predictions size mismatch!"

final_oof_scores = {
    "Recall@5": float(np.mean(aggregated_recalls)),
    "Precision@5": float(np.mean(aggregated_precisions)),
    "MRR@5": float(np.mean(aggregated_mrrs))
}

# Save files to disk
with open(output_results_dir / "link_run.json", "w", encoding="utf-8") as f:
    json.dump(link_run_results, f, indent=2)
with open(output_results_dir / "cv_report.json", "w", encoding="utf-8") as f:
    json.dump({"folds": cv_report, "aggregated_out_of_fold": final_oof_scores}, f, indent=2)

# 4. Select final descriptive implementation alpha using all 40 queries
full_dataset_candidates = get_alpha_candidates(queries)
final_desc_alpha, final_desc_metrics = select_optimal_alpha(queries, full_dataset_candidates)

print("\n--- Cross-Validation Evaluation Summary ---")
print(f"Selection-Aware Out-of-Fold Performance:")
print(f"  Recall@5:    {final_oof_scores['Recall@5']:.4f}")
print(f"  Precision@5: {final_oof_scores['Precision@5']:.4f}")
print(f"  MRR@5:       {final_oof_scores['MRR@5']:.4f}")
print(f"Final Descriptive full-dataset Alpha: {final_desc_alpha:.6f} (MRR@5: {final_desc_metrics[2]:.4f})")

# 5. Update global evaluation scores dictionary for printing
eval_scores["link_expanded"] = final_oof_scores

# Print side-by-side comparison table including Pipeline C
print("\n" + "=" * 65)
print(f"{'Metric':<18} | {'Baseline RAG':<14} | {'Structured RAG':<14} | {'Link-Expanded RAG*':<16}")
print("=" * 65)
for metric in ["Recall@5", "Precision@5", "MRR@5"]:
    b = eval_scores["baseline"][metric]
    s = eval_scores["structured"][metric]
    l = eval_scores["link_expanded"][metric]
    print(f"{metric:<18} | {b:<14.4f} | {s:<14.4f} | {l:<16.4f}")
print("=" * 65)
print("*Note: Link-Expanded RAG score represents the selection-aware out-of-fold performance estimate.")


--- Cross-Validation Evaluation Summary ---
Selection-Aware Out-of-Fold Performance:
  Recall@5:    0.7188
  Precision@5: 0.2300
  MRR@5:       0.7425
Final Descriptive full-dataset Alpha: 0.037861 (MRR@5: 0.7488)

Metric             | Baseline RAG   | Structured RAG | Link-Expanded RAG*
Recall@5           | 0.6958         | 0.6958         | 0.7188          
Precision@5        | 0.2350         | 0.2250         | 0.2300          
MRR@5              | 0.7292         | 0.7438         | 0.7425          
*Note: Link-Expanded RAG score represents the selection-aware out-of-fold performance estimate.


In [ ]:
# ===========================================================================
# 7. Results Freeze Verification Check
# ===========================================================================
from pathlib import Path

remaining = []
for note_path in Path("data/generated_notes").glob("*.md"):
    text = note_path.read_text(encoding="utf-8", errors="ignore")
    if "# Possible Relevance to My Thesis" in text:
        remaining.append(note_path.name)

print("Notes still containing thesis-relevance section:", remaining)
assert not remaining, f"Section still exists in: {remaining}"

Notes still containing thesis-relevance section: []
